# Policy idea extraction

This Snowflake Notebook extracts individual atomic policy ideas from the AI impact survey's `GOVERNMENT_ACTION_SUGGESTION` and `ECONOMIC_IMPACT_EXPECTATION` fields.

Intermediate results are displayed in notebook cells and remain in memory. An optional write cell at the end persists the extracted ideas. The existing policy concept taxonomy is queried for reference but is not used for classification here.

Run cells from top to bottom. Set `ENABLE_OUTPUT_WRITES = True` only when you explicitly want to persist the extracted ideas.

At a high level, the workflow looks like:

1. Extract explicit policy ideas from each relevant phase 1 AI Impact Survey field ("GOVERNMENT_ACTION_SUGGESTION" and/or "ECONOMIC_IMPACT_EXPECTATION").
2. Display the extracted ideas for review.
3. Optionally persist the extracted ideas to Snowflake.


In [ ]:
# Import libraries used for hashing, JSON handling, timestamps, SQL generation, and dataframe display.
import hashlib
import json
import os
import uuid
from datetime import datetime, timezone

import pandas as pd
from dotenv import load_dotenv
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

# Load local environment variables for Snowflake credentials and Cortex model names.
load_dotenv()

# Set pandas display option to show full column width for better readability of long text fields.
pd.set_option('display.max_colwidth', None)

# Use the production dbt source objects currently configured for this analysis.
SOURCE_DATABASE = "TRANSFORM_ENGCA_PRD"
SOURCE_GOVOCAL_SCHEMA = "GOVOCAL"
SOURCE_AI_SCHEMA = "AI_ENGAGEMENT"
SURVEY_TABLE = "INT_GOVOCAL_AI_SURVEY"
TAXONOMY_TABLE = "STG_PHASE2_POLICY_CONCEPTS_AND_THEMES"

# Require an explicit Cortex model configuration.
CORTEX_MODEL = os.environ.get("LLM_MODEL_HIGH", "")

# Record metadata for the current in-memory run.
RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()

# Extract ideas from both survey fields.
EXTRACTION_FIELDS = ["government_action_suggestion", "economic_impact_expectation"]

# Set to True to write outputs.
ENABLE_OUTPUT_WRITES = True
TARGET_DATABASE = "TRANSFORM_ENGCA_DEV"
TARGET_SCHEMA = "DBT_CHOLLINGSWORTH_AI_ENGAGEMENT"
OUTPUT_TABLE = "AI_SURVEY_POLICY_IDEAS"

print(f"run_id={RUN_ID} model={CORTEX_MODEL}")

In [ ]:
def query_df(sql: str, params: list | None = None) -> pd.DataFrame:
    cur.execute(sql, params or [])
    return cur.fetch_pandas_all()


def extract_structured_response(raw: str) -> str:
    payload = json.loads(raw)
    structured = payload.get("structured_output")
    if isinstance(structured, list) and structured:
        first = structured[0]
        raw_message = first.get("raw_message")
        if isinstance(raw_message, dict):
            return json.dumps(raw_message)
        if isinstance(raw_message, str):
            return raw_message
    choices = payload.get("choices")
    if isinstance(choices, list) and choices:
        first_choice = choices[0]
        messages = first_choice.get("messages")
        if isinstance(messages, str):
            return messages
        if isinstance(messages, list):
            content = "".join(msg.get("content", "") for msg in messages if isinstance(msg, dict))
            if content:
                return content
    raise ValueError("Unexpected Snowflake Cortex response shape")


def stable_id(*parts: str) -> str:
    return hashlib.md5("|".join(str(part).strip() for part in parts).encode("utf-8")).hexdigest()


def text_value(value) -> str:
    return "" if pd.isna(value) else str(value).strip()

In [ ]:
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    authenticator=os.environ.get("SNOWFLAKE_AUTHENTICATOR", "externalbrowser"),
    role=os.environ.get("SNOWFLAKE_ROLE", ""),
    warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE", ""),
    database=TARGET_DATABASE,
    schema=TARGET_SCHEMA,
)
cur = conn.cursor()

In [ ]:

SOURCE_SURVEY = f'"{SOURCE_DATABASE}"."{SOURCE_GOVOCAL_SCHEMA}"."{SURVEY_TABLE}"'
SOURCE_TAXONOMY = f'"{SOURCE_DATABASE}"."{SOURCE_AI_SCHEMA}"."{TAXONOMY_TABLE}"'
TARGET_PREFIX = f'"{TARGET_DATABASE}"."{TARGET_SCHEMA}"'

print(SOURCE_SURVEY)
print(SOURCE_TAXONOMY)

In [ ]:
# Query the affinity-mapped taxonomy for reference in later steps.
taxonomy_sql = f"SELECT policy_concept_id, policy_concept, policy_concept_description, subtheme, theme FROM {SOURCE_TAXONOMY}"
taxonomy_df = query_df(taxonomy_sql).fillna("")

print(f"taxonomy concepts: {len(taxonomy_df):,}")
taxonomy_df

In [ ]:
# Define the prompt for extracting explicit atomic policy recommendations.
EXTRACTION_PROMPT = """You extract explicit policy recommendations from one answer to a California public survey, run by Engaged California, about the impact of AI on work.

A recommendation is an action the respondent WANTS a government, employer, school, or other institution to take, or explicitly wants it to refrain from taking. Advocacy markers include: should, must, need to, ought to, imperative verbs ("regulate", "tax them"), and "I want / I'd like the government to". Do NOT create a recommendation from a prediction, fear, complaint, description of impacts, hypothetical ("if the government did X then Y"), or condition ("unless we invest in education") that lacks an advocacy marker.

Rules
1. One idea = one action that could be adopted or rejected on its own. Split compound answers into separate ideas. Do not split an action from its own details or justification.
2. A single word or fragment counts if it names an action ("Regulate.", "UBI", "más capacitación"). Read it in light of the question.
3. stance is "refrain" only when the respondent wants the institution itself to hold back or not intervene ("keep the government out of it", "don't regulate"). A call to ban, stop, prohibit, or limit something is an action the institution should take: stance "do" ("no data centers" -> "Government should not allow data centers.", stance "do").
4. The action must concern AI in a broad sense: AI systems, automation, data centers, the companies that build or deploy AI, or AI's effects on work and workers. Recommendations about unrelated policy (immigration, roads, elections, taxes in general) are not extracted even when phrased as explicit recommendations.
5. "I don't know", jokes, sarcasm, opinions about AI with no action, or off-topic text -> empty ideas list.
6. statement: one English sentence, actor first, present tense, at most 20 words, e.g. "Government should require employers to disclose when AI is used in hiring." Keep the respondent's meaning and specificity; do not add or remove specifics; keep the negation for "refrain".
7. span: an exact contiguous quote copied from the response, in its original language and spelling, that contains this recommendation. Never paraphrase inside span. If the whole response is the recommendation, quote the whole response.

Examples
Response: "Regulate."
-> {"ideas": [{"idea_text": "Government should regulate AI.", "extraction_rationale": "The respondent explicitly advocates for government regulation of AI by stating 'Regulate.'"}]}
Response: "AI will take most jobs within ten years and nothing can stop it."
-> {"ideas": []}
Response: "The economy will grow but inequality will get worse unless we invest in education."
-> {"ideas": []}   (a condition, no advocacy marker)
Response: "Tax companies that replace workers with AI and use it to fund retraining. And stop letting them harvest our data."
-> three ideas:
   "Government should tax companies that replace workers with AI." span "Tax companies that replace workers with AI"
   "Government should fund worker retraining with revenue from taxing AI-driven job replacement." span "use it to fund retraining"
   "Government should stop companies from harvesting personal data." span "stop letting them harvest our data"
Response: "Keep the government out of it, the market will sort it out."
-> one idea: "Government should not regulate AI and should let the market decide."
Response: "Que el gobierno ofrezca cursos gratuitos de IA para trabajadores."
-> {"ideas": [{"idea_text": "Government should offer free AI training courses for workers.", ...}]}
Return JSON only with this shape:
{"ideas": [{"idea_text": "...", "extraction_rationale": "..."}]}
""".strip()

EXTRACTION_PROMPT_VERSION = f"v-{stable_id(EXTRACTION_PROMPT)}"


# Provide the response schema used to request and validate structured Cortex output.
def object_schema(properties: dict, required: list[str]) -> dict:
    return {
        "type": "object",
        "properties": properties,
        "required": required,
        "additionalProperties": False,
    }


# EXTRACTION_SCHEMA = object_schema(
#     {
#         "ideas": {
#             "type": "array",
#             "items": object_schema(
#                 {
#                     "idea_text": {"type": "string"},
#                     "extraction_rationale": {"type": "string"},
#                 },
#                 ["idea_text", "extraction_rationale"],
#             ),
#         }
#     },
#     ["ideas"],
# )


# Normalize extracted ideas and check for empty ideas; malformed responses are retained for review.
def validate_extraction(payload: dict | None) -> list[dict]:
    if not isinstance(payload, dict) or not isinstance(payload.get("ideas"), list):
        raise ValueError("Extraction payload must contain an ideas list")
    ideas = []
    for item in payload["ideas"]:
        if not isinstance(item, dict) or not text_value(item.get("idea_text")):
            raise ValueError("Each extracted idea needs nonempty idea_text")
        ideas.append({"idea_text": text_value(item["idea_text"]), "extraction_rationale": text_value(item.get("extraction_rationale"))})
    return ideas

In [ ]:
# Query published survey responses that contain at least one policy-relevant answer.
survey_sql = f"""
SELECT
    survey_id,
    government_action_suggestion,
    economic_impact_expectation
FROM {SOURCE_SURVEY}
WHERE LOWER(COALESCE(publication_status, 'published')) = 'published'
  AND (
      NULLIF(TRIM(government_action_suggestion), '') IS NOT NULL
      OR NULLIF(TRIM(economic_impact_expectation), '') IS NOT NULL
  )
"""
survey_df = query_df(survey_sql)
print(f"survey responses with relevant text: {len(survey_df):,}")

# Build a normalized extraction input from the published survey response population.
field_sql = {
    "government_action_suggestion": "government_action_suggestion",
    "economic_impact_expectation": "economic_impact_expectation",
}
field_queries = [
    f"SELECT survey_id, '{field}' AS source_field, {field_sql[field]} AS source_text FROM source_survey WHERE NULLIF(TRIM({field_sql[field]}), '') IS NOT NULL"
    for field in EXTRACTION_FIELDS
]
survey_fields_sql = " UNION ALL ".join(field_queries)

# Run extraction for every response in one Snowflake SQL statement. JSON is validated
# in Python so one malformed model response cannot abort the whole query.
extraction_sql = f"""
WITH source_survey AS (
    {survey_sql}
),
survey_fields AS (
    {survey_fields_sql}
)
SELECT
    survey_id,
    source_field,
    source_text,
    SNOWFLAKE.CORTEX.COMPLETE(
        %s,
        ARRAY_CONSTRUCT(
            OBJECT_CONSTRUCT('role', 'system', 'content', %s),
            OBJECT_CONSTRUCT('role', 'user', 'content', CONCAT('Source field: ', source_field, '\\nSurvey response:\\n', source_text))
        ),
        OBJECT_CONSTRUCT(
            'temperature', 0,
            'max_tokens', 4000
        )
    ) AS raw_cortex_response
FROM survey_fields
"""
extraction_df = query_df(
    extraction_sql,
    [CORTEX_MODEL, EXTRACTION_PROMPT],
)

# Validate and flatten each structured response while retaining malformed outputs for review.
def parse_extraction_row(row: pd.Series) -> list[dict]:
    base = {
        "run_id": RUN_ID,
        "run_timestamp": RUN_TIMESTAMP,
        "model_name": CORTEX_MODEL,
        "extraction_prompt_version": EXTRACTION_PROMPT_VERSION,
        "survey_id": row.SURVEY_ID,
        "source_field": row.SOURCE_FIELD,
        "source_text": row.SOURCE_TEXT,
        "source_fingerprint": stable_id(row.SURVEY_ID, row.SOURCE_FIELD, row.SOURCE_TEXT),
        "raw_cortex_response": row.RAW_CORTEX_RESPONSE,
    }
    try:
        raw = row.RAW_CORTEX_RESPONSE
        content = json.loads(extract_structured_response(raw))
        ideas = validate_extraction(content)
        if not ideas:
            return [{**base, "idea_id": stable_id(row.SURVEY_ID, row.SOURCE_FIELD, "NO_IDEA", row.SOURCE_TEXT), "idea_text": "", "extraction_rationale": "", "extraction_status": "no_ideas", "processing_error": None}]
        return [{**base, "idea_id": stable_id(row.SURVEY_ID, row.SOURCE_FIELD, idea["idea_text"]), **idea, "extraction_status": "extracted", "processing_error": None} for idea in ideas]
    except Exception as exc:
        return [{**base, "idea_id": stable_id(row.SURVEY_ID, row.SOURCE_FIELD, "INVALID", row.SOURCE_TEXT), "idea_text": "", "extraction_rationale": "", "extraction_status": "error", "processing_error": str(exc)}]


# Flatten the per-response JSON arrays into one visible dataframe of atomic ideas.
extraction_records = []
for row in extraction_df.itertuples(index=False):
    extraction_records.extend(parse_extraction_row(pd.Series(row._asdict())))
extraction_df = pd.DataFrame(extraction_records)
extraction_df.columns = [c.upper() for c in extraction_df.columns]
print(f"extraction inputs={len(survey_df):,}; extracted result rows={len(extraction_df):,}")
extraction_df

In [ ]:
# Write extracted ideas only when explicitly enabled.
if ENABLE_OUTPUT_WRITES:
    write_pandas(
        conn,
        extraction_df,
        OUTPUT_TABLE,
        database=TARGET_DATABASE,
        schema=TARGET_SCHEMA,
        auto_create_table=True,
        overwrite=True,
        quote_identifiers=True,
    )
    print(f"Wrote {len(extraction_df):,} extraction rows to {TARGET_DATABASE}.{TARGET_SCHEMA}.{OUTPUT_TABLE}.")
else:
    print("Extraction write skipped; set ENABLE_OUTPUT_WRITES = True to opt in.")


# Policy concept classification

The cells below take the extracted policy ideas above and cluster them into substantively-equivalent
policy concepts:

1. Embed each idea and find its nearest-neighbor candidate pairs by embedding similarity.
2. Ask Cortex whether each candidate pair recommends the same governmental action.
3. Build an undirected graph where an edge means "same policy concept" (`YES` classifications only).
4. Run Leiden community detection on that graph; each community is a candidate policy concept, and
   ideas with no `YES` edges remain as their own single-idea concept.
5. Ask Cortex to label and describe each resulting concept.

Every idea (identified by `idea_id`) is kept as a distinct node even when its text duplicates another
idea's text, since downstream analysis counts distinct `survey_id`s per concept. Embeddings and LLM
classifications are cached in Snowflake tables (written only when `ENABLE_OUTPUT_WRITES = True`) so
reruns skip work that's already been done.


In [ ]:
# Configuration for the concept classification pipeline.
N_NEIGHBORS = 25
LEIDEN_RESOLUTION = 1.0
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "snowflake-arctic-embed-l-v2.0-8k")

EMBEDDINGS_TABLE = "AI_SURVEY_POLICY_IDEA_EMBEDDINGS"
PAIRS_TABLE = "AI_SURVEY_POLICY_IDEA_PAIRS"
CLASSIFICATIONS_TABLE = "AI_SURVEY_POLICY_IDEA_PAIR_CLASSIFICATIONS"
CONCEPTS_TABLE = "AI_SURVEY_POLICY_CONCEPTS"
CONCEPT_MEMBERSHIP_TABLE = "AI_SURVEY_POLICY_IDEA_CONCEPTS"

print(f"n_neighbors={N_NEIGHBORS} leiden_resolution={LEIDEN_RESOLUTION} embedding_model={EMBEDDING_MODEL}")

In [ ]:
# Shared helpers for caching intermediate pipeline outputs as Snowflake tables.
def read_cache(table_name: str) -> pd.DataFrame:
    """Return a cached table from the target schema, or an empty frame if it doesn't exist yet."""
    fq = f'"{TARGET_DATABASE}"."{TARGET_SCHEMA}"."{table_name}"'
    try:
        return query_df(f"SELECT * FROM {fq}")
    except snowflake.connector.errors.ProgrammingError:
        return pd.DataFrame()


def write_cache(df: pd.DataFrame, table_name: str, overwrite: bool = False) -> None:
    if not ENABLE_OUTPUT_WRITES or df.empty:
        return
    write_pandas(
        conn, df, table_name, database=TARGET_DATABASE, schema=TARGET_SCHEMA,
        auto_create_table=True, overwrite=overwrite, quote_identifiers=True,
    )


def stage_temp(df: pd.DataFrame, table_name: str) -> str:
    """Write an ephemeral work table (independent of ENABLE_OUTPUT_WRITES) and return its qualified name."""
    write_pandas(
        conn, df, table_name, database=TARGET_DATABASE, schema=TARGET_SCHEMA,
        auto_create_table=True, overwrite=True, quote_identifiers=True, table_type="temporary",
    )
    return f'"{TARGET_DATABASE}"."{TARGET_SCHEMA}"."{table_name}"'


In [ ]:
# Step 0: select the idea population. Duplicate idea_text across responses stays as distinct nodes
# so downstream analysis can count distinct survey_ids per concept.
ideas_df = extraction_df[extraction_df["EXTRACTION_STATUS"] == "extracted"][
    ["IDEA_ID", "IDEA_TEXT", "SURVEY_ID", "SOURCE_FIELD", "SOURCE_TEXT"]
].drop_duplicates("IDEA_ID").reset_index(drop=True)
print(f"policy ideas: {len(ideas_df):,}")
ideas_df


In [ ]:
# Step 1a: embed each idea, reusing any cached embeddings for the current embedding model.
cached_embeddings = read_cache(EMBEDDINGS_TABLE)
if not cached_embeddings.empty:
    cached_embeddings = cached_embeddings[cached_embeddings["EMBEDDING_MODEL"] == EMBEDDING_MODEL]

to_embed = ideas_df[~ideas_df["IDEA_ID"].isin(cached_embeddings.get("IDEA_ID", pd.Series(dtype=str)))]
print(f"ideas needing embeddings: {len(to_embed):,} (cached: {len(cached_embeddings):,})")

if not to_embed.empty:
    work_table = stage_temp(to_embed[["IDEA_ID", "IDEA_TEXT"]], "WORK_IDEAS_TO_EMBED")
    new_embeddings_df = query_df(f"""
        SELECT
            idea_id,
            '{EMBEDDING_MODEL}' AS embedding_model,
            SNOWFLAKE.CORTEX.EMBED_TEXT_1024('{EMBEDDING_MODEL}', idea_text) AS embedding_vector
        FROM {work_table}
    """)
    new_embeddings_df.columns = [c.upper() for c in new_embeddings_df.columns]
else:
    new_embeddings_df = pd.DataFrame(columns=["IDEA_ID", "EMBEDDING_MODEL", "EMBEDDING_VECTOR"])

write_cache(new_embeddings_df, EMBEDDINGS_TABLE)
embeddings_df = pd.concat([cached_embeddings, new_embeddings_df], ignore_index=True).drop_duplicates("IDEA_ID")
embeddings_df = embeddings_df.merge(ideas_df[["IDEA_ID"]], on="IDEA_ID")
print(f"total embeddings available: {len(embeddings_df):,}")


In [ ]:
# Step 1b: build n-nearest-neighbor candidate pairs from embedding similarity, deduping A/B vs B/A.
import numpy as np
from sklearn.neighbors import NearestNeighbors


def to_embedding_matrix(values) -> np.ndarray:
    vectors = []
    for value in values:
        if isinstance(value, np.ndarray):
            vectors.append(value)
        elif isinstance(value, list):
            vectors.append(np.array(value))
        else:
            vectors.append(np.array(json.loads(value)))
    return np.stack(vectors)


ordered = embeddings_df.merge(ideas_df[["IDEA_ID"]], on="IDEA_ID").reset_index(drop=True)
embedding_matrix = to_embedding_matrix(ordered["EMBEDDING_VECTOR"].tolist())
idea_ids = ordered["IDEA_ID"].tolist()

# +1 neighbor accounts for each idea matching itself, which is discarded below.
k = min(N_NEIGHBORS + 1, len(idea_ids))
nn_model = NearestNeighbors(n_neighbors=k, metric="cosine", algorithm="brute").fit(embedding_matrix)
distances, indices = nn_model.kneighbors(embedding_matrix)

pairs = []
for row_pos, idea_id in enumerate(idea_ids):
    for dist, neighbor_pos in zip(distances[row_pos], indices[row_pos]):
        neighbor_id = idea_ids[neighbor_pos]
        if neighbor_id == idea_id:
            continue
        idea_id_a, idea_id_b = sorted((idea_id, neighbor_id))
        pairs.append({
            "IDEA_ID_A": idea_id_a,
            "IDEA_ID_B": idea_id_b,
            "EMBEDDING_SIMILARITY": 1 - float(dist),
            "N_NEIGHBORS": N_NEIGHBORS,
            "EMBEDDING_MODEL": EMBEDDING_MODEL,
        })

candidate_pairs_df = (
    pd.DataFrame(pairs)
    .sort_values("EMBEDDING_SIMILARITY", ascending=False)
    .drop_duplicates(["IDEA_ID_A", "IDEA_ID_B"])
    .reset_index(drop=True)
)
candidate_pairs_df["PAIR_ID"] = candidate_pairs_df.apply(
    lambda r: stable_id(r["IDEA_ID_A"], r["IDEA_ID_B"]), axis=1
)
print(f"candidate pairs: {len(candidate_pairs_df):,}")
write_cache(candidate_pairs_df, PAIRS_TABLE, overwrite=True)
candidate_pairs_df


In [ ]:
# Step 2: ask Cortex whether each candidate pair recommends the same governmental action.
CLASSIFICATION_PROMPT = (
    "You compare two policy recommendations from California state employees and decide if they "
    "represent the SAME policy concept.\n"
    "Same concept: the same governmental/institutional action, even if wording differs substantially, "
    "or only the specific government entity/level differs (ignore that difference).\n"
    "Different concepts: different mechanisms for the same goal; different actions on the same topic; "
    "a general/broad version vs. a specific/narrow version of an action; or one recommendation contains "
    "a materially different, independently-implementable action.\n"
    'Return strict JSON: {"classification": "YES"|"NO", "reason": "<one sentence>"}.'
)
CLASSIFICATION_PROMPT_VERSION = f"v-{stable_id(CLASSIFICATION_PROMPT)}"

cached_classifications = read_cache(CLASSIFICATIONS_TABLE)
if not cached_classifications.empty:
    cached_classifications = cached_classifications[
        (cached_classifications["MODEL_NAME"] == CORTEX_MODEL)
        & (cached_classifications["CLASSIFICATION_PROMPT_VERSION"] == CLASSIFICATION_PROMPT_VERSION)
    ]

to_classify = (
    candidate_pairs_df[~candidate_pairs_df["PAIR_ID"].isin(cached_classifications.get("PAIR_ID", pd.Series(dtype=str)))]
    .merge(ideas_df.rename(columns={"IDEA_ID": "IDEA_ID_A", "IDEA_TEXT": "IDEA_TEXT_A"})[["IDEA_ID_A", "IDEA_TEXT_A"]], on="IDEA_ID_A")
    .merge(ideas_df.rename(columns={"IDEA_ID": "IDEA_ID_B", "IDEA_TEXT": "IDEA_TEXT_B"})[["IDEA_ID_B", "IDEA_TEXT_B"]], on="IDEA_ID_B")
)
print(f"pairs needing classification: {len(to_classify):,} (cached: {len(cached_classifications):,})")

if not to_classify.empty:
    work_table = stage_temp(
        to_classify[["PAIR_ID", "IDEA_ID_A", "IDEA_ID_B", "IDEA_TEXT_A", "IDEA_TEXT_B"]],
        "WORK_PAIRS_TO_CLASSIFY",
    )
    new_classifications_df = query_df(f"""
        SELECT
            pair_id, idea_id_a, idea_id_b, idea_text_a, idea_text_b,
            '{CORTEX_MODEL}' AS model_name,
            '{CLASSIFICATION_PROMPT_VERSION}' AS classification_prompt_version,
            SNOWFLAKE.CORTEX.COMPLETE(
                %s,
                ARRAY_CONSTRUCT(
                    OBJECT_CONSTRUCT('role','system','content', %s),
                    OBJECT_CONSTRUCT('role','user','content',
                        CONCAT('Recommendation A: ', idea_text_a, '\nRecommendation B: ', idea_text_b))
                ),
                OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', 300)
            ) AS raw_cortex_response
        FROM {work_table}
    """, [CORTEX_MODEL, CLASSIFICATION_PROMPT])
    new_classifications_df.columns = [c.upper() for c in new_classifications_df.columns]

    def parse_classification(raw: str) -> dict:
        try:
            content = json.loads(extract_structured_response(raw))
            classification = str(content.get("classification", "")).strip().upper()
            if classification not in {"YES", "NO"}:
                raise ValueError(f"unexpected classification value: {classification!r}")
            return {"CLASSIFICATION": classification, "REASON": text_value(content.get("reason")), "PROCESSING_ERROR": None}
        except Exception as exc:
            return {"CLASSIFICATION": "NO", "REASON": "", "PROCESSING_ERROR": str(exc)}

    parsed = new_classifications_df["RAW_CORTEX_RESPONSE"].apply(parse_classification).apply(pd.Series)
    new_classifications_df = pd.concat([new_classifications_df, parsed], axis=1)
else:
    new_classifications_df = pd.DataFrame(
        columns=["PAIR_ID", "IDEA_ID_A", "IDEA_ID_B", "IDEA_TEXT_A", "IDEA_TEXT_B", "MODEL_NAME", "CLASSIFICATION_PROMPT_VERSION",
                 "RAW_CORTEX_RESPONSE", "CLASSIFICATION", "REASON", "PROCESSING_ERROR"]
    )

if ENABLE_OUTPUT_WRITES:
    classification_cache_fq = f'"{TARGET_DATABASE}"."{TARGET_SCHEMA}"."{CLASSIFICATIONS_TABLE}"'
    for column_name in ("IDEA_TEXT_A", "IDEA_TEXT_B"):
        try:
            cur.execute(f'ALTER TABLE {classification_cache_fq} ADD COLUMN IF NOT EXISTS "{column_name}" TEXT')
        except snowflake.connector.errors.ProgrammingError:
            pass

write_cache(new_classifications_df, CLASSIFICATIONS_TABLE)
classifications_df = pd.concat([cached_classifications, new_classifications_df], ignore_index=True).drop_duplicates("PAIR_ID")
classifications_df = (
    classifications_df.drop(columns=["IDEA_TEXT_A", "IDEA_TEXT_B"], errors="ignore")
    .merge(
        ideas_df.rename(columns={"IDEA_ID": "IDEA_ID_A", "IDEA_TEXT": "IDEA_TEXT_A"})[["IDEA_ID_A", "IDEA_TEXT_A"]],
        on="IDEA_ID_A",
    )
    .merge(
        ideas_df.rename(columns={"IDEA_ID": "IDEA_ID_B", "IDEA_TEXT": "IDEA_TEXT_B"})[["IDEA_ID_B", "IDEA_TEXT_B"]],
        on="IDEA_ID_B",
    )
)
n_errors = classifications_df["PROCESSING_ERROR"].notna().sum()
print(f"total classifications available: {len(classifications_df):,} (parse errors defaulted to NO: {n_errors:,})")
classifications_df

In [ ]:
# Steps 3-4: build the substantive-equivalence graph (YES edges only) and cluster it with Leiden.
import igraph as ig
import leidenalg as la

all_idea_ids = ideas_df["IDEA_ID"].tolist()
id_to_index = {idea_id: i for i, idea_id in enumerate(all_idea_ids)}

edges_df = (
    classifications_df[
        classifications_df["CLASSIFICATION"].eq("YES")
        & classifications_df["IDEA_ID_A"].isin(id_to_index)
        & classifications_df["IDEA_ID_B"].isin(id_to_index)
    ][["IDEA_ID_A", "IDEA_ID_B"]]
)

edge_list = [(id_to_index[a], id_to_index[b]) for a, b in edges_df.itertuples(index=False)]

graph = ig.Graph(n=len(all_idea_ids), edges=edge_list)
# Leiden assigns nodes with no YES edges to their own singleton community, so isolated ideas
# naturally become their own candidate concept without special-casing.
partition = la.find_partition(
    graph, la.RBConfigurationVertexPartition,
    resolution_parameter=LEIDEN_RESOLUTION, seed=42,
)

membership_df = pd.DataFrame({
    "IDEA_ID": all_idea_ids,
    "CONCEPT_ID": partition.membership,
})
concept_sizes = membership_df["CONCEPT_ID"].value_counts()
print(f"concepts: {len(concept_sizes):,}; isolated ideas: {(concept_sizes == 1).sum():,}")
membership_df


In [ ]:
# Step 5: label each multi-idea concept with Cortex; singleton concepts use their one idea directly.
LABELING_PROMPT_HEADER = (
    "The following are California state employee policy recommendations that were grouped because "
    "they represent the SAME underlying governmental action. Return JSON with:\n"
    "- `label`: a concise label (under 50 characters) naming the specific action (not the topic or goal).\n"
    "- `description`: one sentence defining the action represented.\n"
    "Recommendations: "
)
LABELING_PROMPT_VERSION = f"v-{stable_id(LABELING_PROMPT_HEADER)}"
LABELING_RESPONSE_FORMAT = (
    '{"type":"json","schema":{"type":"object",'
    '"properties":{"label":{"type":"string"},"description":{"type":"string"}},'
    '"required":["label","description"],'
    '"additionalProperties":false}}'
)

multi_member = membership_df.groupby("CONCEPT_ID").filter(lambda g: len(g) > 1)
grouped = multi_member.merge(ideas_df, on="IDEA_ID")
concept_texts = grouped.groupby("CONCEPT_ID")["IDEA_TEXT"].apply(lambda s: " | ".join(s)).reset_index()
concept_texts.columns = ["CONCEPT_ID", "IDEA_TEXT"]

if not concept_texts.empty:
    work_table = stage_temp(concept_texts, "WORK_CONCEPTS_TO_LABEL")
    labeled_df = query_df(f"""
        SELECT
            concept_id,
            '{LABELING_PROMPT_VERSION}' AS labeling_prompt_version,
            SNOWFLAKE.CORTEX.COMPLETE(
                %s,
                ARRAY_CONSTRUCT(OBJECT_CONSTRUCT('role','user','content', CONCAT(%s, idea_text))),
                OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', 300, 'response_format', PARSE_JSON(%s))
            ) AS raw_cortex_response
        FROM {work_table}
    """, [CORTEX_MODEL, LABELING_PROMPT_HEADER, LABELING_RESPONSE_FORMAT])
    labeled_df.columns = [c.upper() for c in labeled_df.columns]

    def parse_label(raw: str) -> dict:
        content = json.loads(extract_structured_response(raw))
        return {"CONCEPT_LABEL": text_value(content.get("label")), "CONCEPT_DESCRIPTION": text_value(content.get("description"))}

    labeled_df = pd.concat([labeled_df, labeled_df["RAW_CORTEX_RESPONSE"].apply(parse_label).apply(pd.Series)], axis=1)
    labeled_df = labeled_df[["CONCEPT_ID", "LABELING_PROMPT_VERSION", "CONCEPT_LABEL", "CONCEPT_DESCRIPTION"]]
else:
    labeled_df = pd.DataFrame(columns=["CONCEPT_ID", "LABELING_PROMPT_VERSION", "CONCEPT_LABEL", "CONCEPT_DESCRIPTION"])

singleton_df = membership_df.groupby("CONCEPT_ID").filter(lambda g: len(g) == 1).merge(ideas_df, on="IDEA_ID")
singleton_labels = pd.DataFrame({
    "CONCEPT_ID": singleton_df["CONCEPT_ID"],
    "LABELING_PROMPT_VERSION": LABELING_PROMPT_VERSION,
    "CONCEPT_LABEL": singleton_df["IDEA_TEXT"],
    "CONCEPT_DESCRIPTION": singleton_df["IDEA_TEXT"],
})

concept_labels_df = pd.concat([labeled_df, singleton_labels], ignore_index=True)
write_cache(concept_labels_df, CONCEPTS_TABLE, overwrite=True)
concept_labels_df

In [ ]:
# Step 6: classify each policy concept into one existing taxonomy subtheme.
SUBTHEME_CLASSIFICATIONS_TABLE = "AI_SURVEY_POLICY_CONCEPT_SUBTHEMES"

# Give Cortex the existing subthemes and representative taxonomy concepts as the only allowed choices.
taxonomy_subthemes = (
    taxonomy_df[["SUBTHEME", "POLICY_CONCEPT", "POLICY_CONCEPT_DESCRIPTION"]]
    .dropna(subset=["SUBTHEME"])
    .assign(SUBTHEME=lambda df: df["SUBTHEME"].map(text_value))
)
taxonomy_subthemes = taxonomy_subthemes[taxonomy_subthemes["SUBTHEME"] != ""]
subtheme_options = (
    taxonomy_subthemes.groupby("SUBTHEME", sort=True)
    .apply(
        lambda group: "\n".join(
            f"- {group.name}: {text_value(row['POLICY_CONCEPT'])} - {text_value(row['POLICY_CONCEPT_DESCRIPTION'])}"
            for _, row in group.iterrows()
        ),
        include_groups=False,
    )
    .to_dict()
)
allowed_subthemes = set(subtheme_options)
subtheme_reference = "\n".join(
    f"{subtheme}:\n{examples}" for subtheme, examples in subtheme_options.items()
)

SUBTHEME_CLASSIFICATION_PROMPT = f"""Classify each policy concept into exactly one existing subtheme from the taxonomy.
Use the concept label and description to choose the best substantive match. Return only one of the listed subtheme names.
Do not create a new subtheme, combine subthemes, or classify based only on shared words.

Existing taxonomy subthemes and representative concepts:
{subtheme_reference}

Return strict JSON: {{"subtheme": "<exact existing subtheme name>", "reason": "<one sentence>"}}"""
SUBTHEME_RESPONSE_FORMAT = json.dumps({
    "type": "json",
    "schema": {
        "type": "object",
        "properties": {
            "subtheme": {"type": "string", "enum": sorted(allowed_subthemes)},
            "reason": {"type": "string"},
        },
        "required": ["subtheme", "reason"],
        "additionalProperties": False,
    },
})
SUBTHEME_CLASSIFICATION_PROMPT_VERSION = f"v-{stable_id(SUBTHEME_CLASSIFICATION_PROMPT, SUBTHEME_RESPONSE_FORMAT)}"

cached_subthemes = read_cache(SUBTHEME_CLASSIFICATIONS_TABLE)
if not cached_subthemes.empty:
    if "SUBTHEME_CLASSIFICATION_PROMPT_VERSION" not in cached_subthemes.columns:
        cached_subthemes = pd.DataFrame()
    else:
        cached_subthemes = cached_subthemes[
            (cached_subthemes["MODEL_NAME"] == CORTEX_MODEL)
            & (cached_subthemes["SUBTHEME_CLASSIFICATION_PROMPT_VERSION"] == SUBTHEME_CLASSIFICATION_PROMPT_VERSION)
        ]

concepts_to_classify = concept_labels_df[
    ~concept_labels_df["CONCEPT_ID"].isin(cached_subthemes.get("CONCEPT_ID", pd.Series(dtype=str)))
].copy()
print(f"concepts needing subtheme classification: {len(concepts_to_classify):,} (cached: {len(cached_subthemes):,})")

if not concepts_to_classify.empty:
    work_table = stage_temp(
        concepts_to_classify[["CONCEPT_ID", "CONCEPT_LABEL", "CONCEPT_DESCRIPTION"]],
        "WORK_CONCEPTS_TO_SUBTHEME",
    )
    new_subthemes_df = query_df(f"""
        SELECT
            concept_id,
            '{CORTEX_MODEL}' AS model_name,
            '{SUBTHEME_CLASSIFICATION_PROMPT_VERSION}' AS subtheme_classification_prompt_version,
            SNOWFLAKE.CORTEX.COMPLETE(
                %s,
                ARRAY_CONSTRUCT(OBJECT_CONSTRUCT(
                    'role', 'user',
                    'content', CONCAT(
                        %s,
                        '\n\nConcept label: ', concept_label,
                        '\nConcept description: ', concept_description
                    )
                )),
                OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', 200, 'response_format', PARSE_JSON(%s))
            ) AS raw_cortex_response
        FROM {work_table}
    """, [CORTEX_MODEL, SUBTHEME_CLASSIFICATION_PROMPT, SUBTHEME_RESPONSE_FORMAT])
    new_subthemes_df.columns = [c.upper() for c in new_subthemes_df.columns]

    def parse_subtheme(raw: str) -> dict:
        try:
            content = json.loads(extract_structured_response(raw))
            subtheme = text_value(content.get("subtheme"))
            if subtheme not in allowed_subthemes:
                raise ValueError(f"unexpected subtheme value: {subtheme!r}")
            return {"SUBTHEME": subtheme, "REASON": text_value(content.get("reason")), "PROCESSING_ERROR": None}
        except Exception as exc:
            return {"SUBTHEME": None, "REASON": "", "PROCESSING_ERROR": str(exc)}

    parsed_subthemes = new_subthemes_df["RAW_CORTEX_RESPONSE"].apply(parse_subtheme).apply(pd.Series)
    new_subthemes_df = pd.concat([new_subthemes_df, parsed_subthemes], axis=1)
else:
    new_subthemes_df = pd.DataFrame(
        columns=["CONCEPT_ID", "MODEL_NAME", "SUBTHEME_CLASSIFICATION_PROMPT_VERSION", "RAW_CORTEX_RESPONSE",
                 "SUBTHEME", "REASON", "PROCESSING_ERROR"]
    )

write_cache(new_subthemes_df, SUBTHEME_CLASSIFICATIONS_TABLE, overwrite=True)
concept_subthemes_df = (
    pd.concat([cached_subthemes, new_subthemes_df], ignore_index=True)
    .drop_duplicates("CONCEPT_ID")
)
print(f"total concept subthemes available: {len(concept_subthemes_df):,}")
concept_subthemes_df[["CONCEPT_ID", "SUBTHEME", "PROCESSING_ERROR"]]

In [ ]:
# Final table: one row per policy idea with its resolved concept label/description and subtheme.
final_df = (
    membership_df.merge(ideas_df, on="IDEA_ID")
    .merge(concept_labels_df, on="CONCEPT_ID")
    .merge(concept_subthemes_df[["CONCEPT_ID", "SUBTHEME"]], on="CONCEPT_ID", how="left")
    .rename(columns={
        "IDEA_ID": "POLICY_IDEA_ID",
        "IDEA_TEXT": "POLICY_IDEA",
        "CONCEPT_ID": "POLICY_CONCEPT_ID",
        "CONCEPT_LABEL": "POLICY_CONCEPT_LABEL",
        "CONCEPT_DESCRIPTION": "POLICY_CONCEPT_DESCRIPTION",
        "SUBTHEME": "POLICY_SUBTHEME",
    })
    [["SURVEY_ID", "SOURCE_FIELD", "SOURCE_TEXT", "POLICY_IDEA_ID", "POLICY_IDEA", "POLICY_CONCEPT_ID", "POLICY_CONCEPT_LABEL", "POLICY_CONCEPT_DESCRIPTION", "POLICY_SUBTHEME"]]
)

# Quality-control summary.
n_yes = (classifications_df["CLASSIFICATION"] == "YES").sum()
n_no = (classifications_df["CLASSIFICATION"] == "NO").sum()
n_total_classified = n_yes + n_no
print(f"ideas: {len(ideas_df):,}")
print(f"candidate pairs: {len(candidate_pairs_df):,}")
print(f"YES: {n_yes:,} ({n_yes / n_total_classified:.1%})  NO: {n_no:,} ({n_no / n_total_classified:.1%})")
print(f"policy concepts: {len(concept_sizes):,}")
print(f"isolated ideas (singleton concepts): {(concept_sizes == 1).sum():,}")
print(f"subtheme classifications: {concept_subthemes_df['SUBTHEME'].notna().sum():,}")
print("concept size distribution:")
print(concept_sizes.describe())

final_df


In [ ]:
# Write the final idea-to-concept mapping only when explicitly enabled.
if ENABLE_OUTPUT_WRITES:
    write_pandas(
        conn,
        final_df,
        CONCEPT_MEMBERSHIP_TABLE,
        database=TARGET_DATABASE,
        schema=TARGET_SCHEMA,
        auto_create_table=True,
        overwrite=True,
        quote_identifiers=True,
    )
    print(f"Wrote {len(final_df):,} rows to {TARGET_DATABASE}.{TARGET_SCHEMA}.{CONCEPT_MEMBERSHIP_TABLE}.")
else:
    print("Concept membership write skipped; set ENABLE_OUTPUT_WRITES = True to opt in.")


In [ ]:
# Close the local cursor and connection after all desired cells have been inspected.
cur.close()
conn.close()
print("Snowflake connection closed.")